# Page 6 — "Thinking Longer": the length of the chain is the compute

Run this, do not just read it. Every number and every trace on page 6 comes from here; the page
renders committed logs and runs no model of its own.

**The experiment.** `max_new_tokens` is the budget dial. Cap the model, make it commit at the cap,
grade what it commits to, sweep the cap. Three beats come out of one sweep:

1. cut one trace at a budget and watch the answer flip;
2. the same cut across every problem, giving a climb that bends over;
3. each problem's *flip point*, the budget where it first grades correct, including the band that
   never does.

**Decoding is greedy throughout**, so budget is the only variable. Sampling is page 7's subject.

**From the book we take the harness only** ([`https://github.com/rasbt/reasoning-from-scratch`](../https://github.com/rasbt/reasoning-from-scratch)): `load_math500_test`,
`render_prompt`, `extract_final_candidate`, `grade_answer`, `get_device`. No checkpoint and no
reported number.

Why the page is built this way, and what it may and may not claim from this run, live in
[`docs/page_6_refactor.md`](../docs/page_6_refactor.md). This file is the procedure.


## 0. How to run this

Two modes, set by one variable in the next cell.

| Mode | Model(s) | Items | Budget grid | Where it finishes |
|---|---|---|---|---|
| `QUICK` | Qwen3-0.6B | 20 | 32 · 64 · 128 · 256 | a laptop CPU, ~20 min |
| `FULL` | Qwen3-4B + Qwen3-1.7B | 160 | 32 · 64 · 128 · 256 · 512 · 1024 · 2048 | a 16 GB GPU, ~4-6 h |

Those times are estimates until someone runs it; the loading cell measures tokens per second on
this machine and section 8 projects from that.

`QUICK`'s grid is the **left half of `FULL`'s**, so the x-axis means the same thing in both. It
demonstrates the *climb* and produces every aggregate figure, but stopping at 256 tokens it may not
reach the *bend*, which is what `FULL` buys. The page ships `FULL`.

**If the model does not fit:** the ladder is 4B → 1.7B → 0.6B, each roughly halving the runtime
above it, all hybrid Qwen3 checkpoints sharing one tokenizer so the x-axis survives the swap. Past
that, set `LOAD_IN_4BIT = True` in section 3 (`pip install bitsandbytes`) and expect the same shape
at a lower level, though it puts quantisation inside our own numbers so the page should ship bf16.

**`transformers >= 4.56` is a hard pin.** Before it the argument was spelled `torch_dtype=`, and an
older version silently loads fp32 while the manifest records bf16. Section 3 checks the dtype the
weights actually have and stops if it is wrong.

**Runs are resumable.** Every generation is appended to `out/page_06/traces.jsonl` as it happens
and reloaded next time. Delete that file to start over.


In [ ]:
# ---------------------------------------------------------------------------
# The two knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (laptop, ~20 min) or "FULL" (16 GB GPU, hours)
SEED = 123                # only selects items; decoding is greedy and unseeded

PRESETS = {
    "QUICK": dict(
        model_id="Qwen/Qwen3-0.6B",
        second_model_id=None,                    # one curve in QUICK; figure 2 wants two
        n_items=20,
        budgets=[32, 64, 128, 256],
    ),
    "FULL": dict(
        model_id="Qwen/Qwen3-4B",                # primary curve
        second_model_id="Qwen/Qwen3-1.7B",       # second curve, same family, one size down
        n_items=160,
        budgets=[32, 64, 128, 256, 512, 1024, 2048],
    ),
}
CFG = PRESETS[RUN_MODE]

# Log-spaced, because that is the axis figure 2 is drawn on. Doubling is the unit
# of the argument: the question is not "does more help" but "what does the next
# doubling buy", and that only reads properly on a log x-axis.
BUDGETS = sorted(CFG["budgets"])
MAX_BUDGET = BUDGETS[-1]

# Tokens the model gets AFTER we close its think block and hand it "\boxed{".
# Enough to write an answer and shut the brace, not enough to resume reasoning.
# 32 rather than 16 because MATH-500 answers at levels 4-5 are often not a bare
# number: \left( 3, \frac{\pi}{2} \right) is about fifteen tokens by itself, and a
# brace that never closes is graded by a different rule (see section 10, check 2).
# Section 10 reports how often it was still too few.
ANSWER_ALLOWANCE = 32

# Same family for both curves on purpose: token counts are tokenizer-specific, so
# plotting two families on a shared token-budget axis would be apples to oranges.
# Same family means the x-axis means one thing and size is the only variable.

print(f"mode={RUN_MODE}  model={CFG['model_id']}"
      f"{' + ' + CFG['second_model_id'] if CFG['second_model_id'] else ''}")
print(f"items={CFG['n_items']}  budgets={BUDGETS}  answer allowance={ANSWER_ALLOWANCE}")
print(f"worst-case thinking tokens per problem: {sum(BUDGETS):,}")


In [ ]:
import gc
import hashlib
import json
import random
import subprocess
import textwrap
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# The inherited harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_06")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = lambda xs: sum(xs) / max(len(xs), 1)

print("torch", torch.__version__)


## 1. The grading harness we inherit

Grading free-form maths answers is fiddly and easy to get subtly wrong, so we do not write our own.
`extract_final_candidate` pulls the last `\boxed{...}` out of a completion (falling back to the last
number if the model never boxed anything), and `grade_answer` compares it to ground truth with light
LaTeX canonicalisation and a SymPy equivalence check.

Using the book's grader unchanged means **the grading is not ours to defend** — if you dispute a
score, the argument is with a published, separately-tested function rather than with us.

Page 6 leans on it harder than page 5 did, because our intervention hands the model a `\boxed{`
opening brace and lets it fill in the rest. So the checks below add the two shapes this page
actually produces: a trace cut mid-sentence and then forced to commit, and a trace cut mid-sentence
with **no** forcing at all. The second is why section 5 forces: without it, low budgets grade as "no
answer", the curve starts at zero for a trivial reason, and the page measures nothing.


In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                             "9",               True),
    (r"27",                                                    "9",               False),  # wrong blurt
    (r"\boxed{\frac{1}{2}}",                                  "0.5",             True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                        "1024",            True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",              r"\left( 3, \frac{\pi}{2} \right)", True),

    # --- the two shapes page 6 generates -----------------------------------
    # cut mid-sentence, think block closed for it, then forced to commit:
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Adding gives 17/12. "
     "Now I need to subtr\n</think>\n\nFinal answer: " + r"\boxed{\frac{17}{12}}",
     r"\frac{17}{12}",  True),
    # cut mid-sentence with no forcing: nothing to extract, so it must grade wrong.
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Now I need to subtr",
     r"\frac{17}{12}",  False),
    # forced, but the model committed to the wrong thing -- forcing must not rescue it.
    ("...\n</think>\n\nFinal answer: " + r"\boxed{\frac{5}{12}}", r"\frac{17}{12}", False),
]

rows = [(raw, gt, expected, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, expected in CHECKS]
failures = [r for r in rows if r[3] != r[2]]

if failures:                                   # the table is for debugging, not for reading
    for raw, gt, expected, got in rows:
        flag = "ok " if got == expected else "!! "
        shown = raw if len(raw) < 58 else raw[:30] + " … " + raw[-25:]
        print(f"{flag}{shown:<60} -> {extract_final_candidate(raw)!r:<22} vs {gt!r:<20} {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")

print(f"grader preflight: {len(rows)}/{len(rows)} as expected, including the two shapes this page "
      f"generates (cut then forced; cut with nothing to extract)")


## 2. The item set

MATH-500, levels 4 and 5 only. The figures only exist if the model fails at low budget and succeeds
at high budget, and the full set sits near a maths-tuned model's ceiling; [`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §3 argues the
headroom constraint. Section 7's pilot measures it on this model rather than assuming it.

Items are drawn with a fixed seed and hashed, so "the same items" is checkable later rather than
asserted. The seed selects items and nothing else: decoding is greedy, so there is no sampling for
a seed to control.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

hard = [r for r in math500 if int(r["level"]) in (4, 5)]
hard.sort(key=lambda r: r["unique_id"])          # deterministic before shuffling

rng = random.Random(SEED)
order = list(range(len(hard)))
rng.shuffle(order)
items = [hard[i] for i in order[: CFG["n_items"]]]

if len(items) < CFG["n_items"]:
    raise RuntimeError(f"asked for {CFG['n_items']} items, levels 4-5 hold only {len(hard)}")


def sha256_of(strings):
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()


ITEM_SET_HASH = sha256_of(r["unique_id"] for r in items)

from collections import Counter
band = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"scored {len(items)} of {len(hard)} available at levels 4-5   {band}   "
      f"sha256 {ITEM_SET_HASH[:12]}")


## 3. The models

Qwen3, because this page needs a model that blurts when not asked to think *and* thinks on demand
with the thinking separable enough to cut. Only hybrid models do both, and [`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §3 works through
the alternatives that were ruled out.

One consequence is load-bearing **here** rather than in the spec: both curves come from the same
family, so they share a tokenizer. That is why section 4 derives the `</think>` and end-of-turn ids
once and reuses them for both models, and it is the first thing that breaks if the ladder ever
leaves the Qwen3 family.

Precision is bf16 on GPU and fp32 on CPU, and the cell below checks the dtype the weights actually
loaded with rather than the one we asked for.


In [ ]:
device = get_device()
DTYPE = torch.float32 if device.type == "cpu" else torch.bfloat16

LOAD_IN_4BIT = False       # last resort for small VRAM; see section 0


WARMUP_TOK_S = None            # measured below; section 8 projects the sweep from it


def load_model(model_id):
    # from_pretrained only draws a progress bar for multi-shard checkpoints, and these
    # are single-file, so the cell would otherwise sit blank with no way to tell
    # downloading from loading from hung. Fetch first, then load, each timed.
    print(f"[1/4] fetch      {model_id}")
    t0 = time.time()
    model_id = snapshot_download(model_id)         # draws its own per-file bars; cached
    print(f"      {time.time() - t0:>6.1f}s")

    print("[2/4] load       tokenizer and weights")
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id)

    kwargs = {}
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig          # pip install bitsandbytes
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    else:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id, dtype=DTYPE, **kwargs).to(device).eval()

    # `dtype=` is the transformers >= 4.56 spelling (it was `torch_dtype` before).
    # On an older version the kwarg is swallowed into the config and the model loads
    # in fp32 without complaint -- which OOMs a 16 GB card at 4B, and would leave the
    # manifest below asserting bf16 for a run that never used it. So check what we
    # actually got rather than what we asked for.
    got = next(mdl.parameters()).dtype
    if not LOAD_IN_4BIT and got != DTYPE:
        raise RuntimeError(
            f"asked for {DTYPE} but the weights loaded as {got}. Almost certainly "
            f"transformers < 4.56, where the argument is spelled torch_dtype= and "
            f"dtype= is silently ignored. Upgrade, or rename the argument.")

    print(f"      {time.time() - t0:>6.1f}s")

    # A short generation before the sweep. It absorbs the one-off kernel-compilation and
    # cache-allocation cost that would otherwise be charged to the first real item, and it
    # measures tokens per second on THIS machine, which beats the estimates in section 0.
    global WARMUP_TOK_S
    print("[3/4] warm up    16 tokens")
    t0 = time.time()
    with torch.inference_mode():
        mdl.generate(**tok("Hello", return_tensors="pt").to(mdl.device),
                     max_new_tokens=16, do_sample=False,
                     pad_token_id=tok.pad_token_id if tok.pad_token_id is not None
                     else tok.eos_token_id)
    dt = time.time() - t0
    WARMUP_TOK_S = 16 / max(dt, 1e-6)
    print(f"      {dt:>6.1f}s  ~{WARMUP_TOK_S:.1f} tokens/s")

    n = sum(p.numel() for p in mdl.parameters())
    print(f"[4/4] ready      {n/1e9:.2f}B params  {got}  on {device}\n")
    return mdl, tok, n


def free_memory():
    '''FULL loads a 4B and a 1.7B; on 16 GB they do not want to be resident at the
    same time, so each model is swept and then deleted. Note the deletion has to
    happen at the call site -- handing the model to a function that does `del` on
    its own parameter frees nothing, because the caller still holds a reference.'''
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        torch.mps.empty_cache()


model, tokenizer, N_PARAMS = load_model(CFG["model_id"])
PRIMARY_LOADED = True          # tracked so section 10 knows whether it can re-run a generation


## 4. Where the think block is

This is the cell to read if you suspect we cheated.

Qwen3's chat template takes an `enable_thinking` flag. **Page 5 turned it off**, so the prompt was
the only thing varying. **Page 6 turns it on**, because the think block is the thing we cut. With
the flag on the template hands over with nothing pre-filled and the model opens `<think>` itself;
with it off the template pre-closes the block, leaving nowhere to reason.

The templates are identical until the very end, so only the tail is printed, with newlines shown
as `\n` because the difference is a newline arrangement and `repr` would bury it.

Three properties the intervention depends on, asserted rather than assumed: `</think>` is a real
single token, so "did it close the block" is a question about token positions; it is an *ordinary*
token, so it survives `skip_special_tokens=True` and we can still see what we are cutting; and the
end-of-turn ids are known, so "the model stopped on its own" is distinguishable from "we stopped
it". That last distinction is the whole difference between a natural trace and a forced one.


In [ ]:
demo_msgs = [{"role": "user", "content": "What is 2+2?"}]
on = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=True)
off = tokenizer.apply_chat_template(demo_msgs, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)

# Everything before the first difference is identical boilerplate, so print from there.
d = 0
while d < min(len(on), len(off)) and on[d] == off[d]:
    d += 1
show = lambda s: s[d:].replace("\n", "\\n")     # the difference IS whitespace, so show it

print(f"both templates agree for {d} characters, ending: ...{on[max(0, d - 30):d]!r}")
print()
print(f"  ON  (used here)  ...{show(on)}")
print(f"  OFF (page 5)     ...{show(off)}")
print("\nON hands over with nothing pre-filled, so the model opens <think> itself and we")
print("have a block to cut. OFF pre-closes it, which is what page 5 wanted and this page")
print("cannot use.")

THINK_OPEN_ID = tokenizer.convert_tokens_to_ids("<think>")
THINK_CLOSE_ID = tokenizer.convert_tokens_to_ids("</think>")

assert THINK_CLOSE_ID is not None and THINK_CLOSE_ID != tokenizer.unk_token_id, (
    "this checkpoint has no </think> token, so it is not a hybrid thinking model and the "
    "intervention in section 5 does not apply to it")
assert "</think>" in tokenizer.decode([THINK_CLOSE_ID], skip_special_tokens=True), (
    "this checkpoint marks </think> as a *special* token, so skip_special_tokens=True erases "
    "the boundary we cut at; decode with skip_special_tokens=False and filter the EOS ids by hand")

# Only the ids `generate` actually stops on. Widening this set with the tokenizer's
# own eos would be a guess, and a wrong guess here silently turns a cut trace into a
# finished one.
gen_eos = model.generation_config.eos_token_id
EOS_IDS = set(gen_eos) if isinstance(gen_eos, (list, tuple)) else {gen_eos}
EOS_IDS.discard(None)
if tokenizer.eos_token_id not in EOS_IDS:
    print(f"note: tokenizer eos ({tokenizer.eos_token_id}) is not a generation stop id; "
          f"using the generation config's set")

print(f"\n<think>  id={THINK_OPEN_ID}   </think> id={THINK_CLOSE_ID}   (ordinary tokens, kept on decode)")
print(f"end-of-turn ids: {sorted(EOS_IDS)}")
assert "</think>" not in on, "thinking-on unexpectedly pre-closes the block; there is nothing to cut"
assert "</think>" in off, "thinking-off does not pre-close the block on this checkpoint"


## 5. The intervention: cap, close, force, grade

This is the cell the reader came for, so it is here rather than behind an import. Four steps, each
one a line you can point at.

1. **Cap.** Generate at most `budget` tokens. `max_new_tokens` *is* the dial.
2. **Close.** If the model ran out of room while still inside `<think>`, inject `</think>`. It
   already knows what that token means, so this is a signal rather than a string splice.
3. **Force.** Append `Final answer: \boxed{` and let it generate a few more tokens. Without this,
   every low budget grades as "no answer" and the curve is zero for a reason that has nothing to do
   with reasoning. We stop as soon as the brace we opened closes.
4. **Grade.** The inherited `extract_final_candidate` then `grade_answer`, unchanged.

**When the cap does not bind, we do not intervene.** A model that finishes inside its budget is
recorded as a plain unforced generation. Forcing happens only where the dial actually bites, and
section 10 reports how often that was.

**Three fields make figure 1 drawable:** `tokens` (the trace as token strings, because the page
reveals it one real token at a time), `cut_index` (where the model's own words end) and
`answer_start` (where it resumes after our injection). Note what that means: the span between them
is *our* tokenisation of text the model never emitted, so those indices bracket an intervention
rather than a generation. That is why the page tints it differently.

The two assertions in the code are not decoration. Byte-level BPE can split one character across
two tokens, and the resulting corruption is silent, so the trace is decoded twice by independent
routes and the results are required to match.


In [ ]:
DECODE_WINDOW = 16          # tokens of look-back; a character spans 2-3 at most


def tokenizer_fingerprint(tok):
    '''Identifies the exact tokenizer. Token counts are the x-axis of this whole
    page, so "same tokenizer" has to be checkable rather than assumed.'''
    try:
        return hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
    except Exception:
        return None


def incremental_token_strings(ids, tok):
    '''One string per token, such that "".join(pieces) reproduces tok.decode(ids).

    Decoding tokens one at a time is not safe. Qwen3 uses byte-level BPE, so a single
    character -- a degree sign, a pi, a CJK glyph -- can be split across two tokens,
    and decoding half of one yields the replacement character U+FFFD. So a trailing
    U+FFFD is treated as "not settled yet": the token that starts such a character
    contributes nothing and the token that completes it contributes the whole thing.
    Without this the text silently corrupts, and it is the text that gets graded.

    The decode looks back a fixed window rather than over the whole prefix, which
    keeps this linear in the length of the trace instead of quadratic. Callers assert
    the join against an independent full decode, so any error here is loud.'''
    pieces = []
    for k in range(len(ids)):
        lo = max(0, k - DECODE_WINDOW)
        prev = tok.decode(ids[lo:k], skip_special_tokens=True).rstrip("\ufffd")
        cur = tok.decode(ids[lo:k + 1], skip_special_tokens=True).rstrip("\ufffd")
        pieces.append(cur[len(prev):] if cur.startswith(prev) else "")
    return pieces


def cut_at_boxed_close(pieces):
    '''We prefilled one open brace with "\\boxed{". Walk the model's continuation
    until that brace closes, tracking depth so nested LaTeX like \\frac{1}{2}
    is not cut in half. Returns (kept token strings, did it close).'''
    depth, kept = 1, []
    for p in pieces:
        out = ""
        for ch in p:
            out += ch
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    kept.append(out)              # may end mid-token, on purpose
                    return kept, True
        kept.append(out)
    return kept, False


@torch.inference_mode()
def continue_ids(ids, mdl, tok, max_new_tokens):
    '''Greedy continuation from an explicit id sequence -- this is the prefill that
    budget forcing needs and that a hosted chat API will not reliably give you.
    Returns the newly generated ids only.'''
    inp = torch.tensor([ids], device=mdl.device)
    out = mdl.generate(
        input_ids=inp,
        attention_mask=torch.ones_like(inp),
        max_new_tokens=max_new_tokens,
        # Qwen3 ships do_sample=True, temperature=0.6, top_p=0.95, top_k=20 in its
        # generation_config. Turning do_sample off is not enough -- the others have
        # to be cleared too, or the manifest below records a decoding setup that was
        # never actually configured.
        do_sample=False,
        temperature=None, top_p=None, top_k=None,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
    )
    return out[0][inp.shape[1]:].tolist()


In [ ]:
FORCE_CLOSE = "\n</think>\n\n"          # only injected if the model never closed the block
FORCE_STEM = "Final answer: \\boxed{"    # the brace we open and it must close


def run_at_budget(question, budget, mdl, tok, allowance=ANSWER_ALLOWANCE):
    chat = tok.apply_chat_template(
        [{"role": "user", "content": render_prompt(question)}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=True,                     # <- load-bearing, see section 4
    )
    prompt_ids = tok(chat, add_special_tokens=False).input_ids

    # --- 1. cap -------------------------------------------------------------
    gen_ids = continue_ids(prompt_ids, mdl, tok, budget)

    # Whether the cap bound is decided by the count alone. Testing the last token
    # against an end-of-turn id would be a guess about which ids `generate` stops
    # on, and getting that wrong marks a cut trace as a finished one -- which then
    # propagates through the carry-forward in section 6 and fabricates the top of
    # the curve. The count cannot be wrong: generate returns fewer than the cap
    # only if it stopped on its own.
    hit_cap = len(gen_ids) >= budget
    while gen_ids and gen_ids[-1] in EOS_IDS:
        gen_ids.pop()                             # end-of-turn is not visible content

    own_text = tok.decode(gen_ids, skip_special_tokens=True)      # decoded independently
    own_pieces = incremental_token_strings(gen_ids, tok)
    assert "".join(own_pieces) == own_text, (
        "token strings do not reconstruct the decoded text -- see incremental_token_strings; "
        "the page animates these token by token, so a mismatch would put a lie on the screen")

    # By token id, not by string search: </think> is a single token, so "did it close
    # the block" is a question about token positions.
    think_closed = THINK_CLOSE_ID in gen_ids
    n_thinking = gen_ids.index(THINK_CLOSE_ID) + 1 if think_closed else len(gen_ids)

    if not hit_cap:
        # The dial is not binding here. No intervention: this is simply what the
        # model does when it has more room than it needs. Note that gen_ids holds
        # the thinking AND the model's own answer, so the two counts differ.
        return dict(trace=own_text, tokens=own_pieces,
                    thinking_tokens=n_thinking, answer_tokens=0,
                    tokens_generated=len(gen_ids),
                    truncated=False, forced=False, forced_suffix="",
                    think_closed=think_closed, answer_closed=None,
                    cut_index=None, answer_start=None)

    # --- 2. close + 3. force -----------------------------------------------
    suffix = (FORCE_CLOSE if not think_closed else "\n\n") + FORCE_STEM
    suffix_ids = tok(suffix, add_special_tokens=False).input_ids
    suffix_pieces = incremental_token_strings(suffix_ids, tok)
    assert "".join(suffix_pieces) == suffix, (
        "the forced suffix does not survive a tokenise/detokenise round trip on this "
        "checkpoint; figure 1 would draw the intervention in the wrong place")

    ans_ids = continue_ids(prompt_ids + gen_ids + suffix_ids, mdl, tok, allowance)
    while ans_ids and ans_ids[-1] in EOS_IDS:
        ans_ids.pop()
    ans_pieces = incremental_token_strings(ans_ids, tok)
    assert "".join(ans_pieces) == tok.decode(ans_ids, skip_special_tokens=True)
    kept, closed = cut_at_boxed_close(ans_pieces)   # truncated at the closing brace

    tokens = own_pieces + suffix_pieces + kept
    return dict(trace="".join(tokens), tokens=tokens,
                thinking_tokens=n_thinking, answer_tokens=len(kept),
                tokens_generated=len(gen_ids) + len(kept),   # our suffix is not the model's cost
                truncated=True, forced=True, forced_suffix=suffix,
                think_closed=think_closed, answer_closed=closed,
                cut_index=len(own_pieces),
                answer_start=len(own_pieces) + len(suffix_pieces))


In [ ]:
def render_trace(g, width=94, head=6, tail=3):
    '''Print a trace the way the page draws it: the model's own words, the point
    where the budget ran out, the suffix we injected, and what it committed to.'''
    own = "".join(g["tokens"][: g["cut_index"]]) if g["forced"] else g["trace"]
    lines = textwrap.wrap(own.strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"… [{len(lines) - head - tail} lines elided] …"] + lines[-tail:]
    out = [f"  {ln}" for ln in lines]
    if g["forced"]:
        # The cut lands exactly on a token boundary -- cut_index is a token index --
        # which usually means mid-word, and often mid-sentence. That is the pixel
        # figure 1 exists to show.
        out[-1] += " ▌"
        out.append("  " + "┄" * (width // 2) + "  budget spent, block closed for it")
        forced_text = "".join(g["tokens"][g["answer_start"]:])
        out.append(f"  >> Final answer: \\boxed{{{forced_text}"
                   f"{'' if g['answer_closed'] else '   (never closed the brace)'}")
    return "\n".join(out)


demo = items[0]
print(textwrap.fill("Q: " + demo["problem"], 96))
print("ground truth:", demo["answer"], "\n")

for b in (BUDGETS[0], BUDGETS[len(BUDGETS) // 2], MAX_BUDGET):
    g = run_at_budget(demo["problem"], b, model, tokenizer)
    assert "".join(g["tokens"]) == g["trace"], "token strings do not reconstruct the text"
    got = extract_final_candidate(g["trace"])
    ok = bool(grade_answer(got, demo["answer"]))
    print("=" * 100)
    print(f"budget {b:>5}   thinking {g['thinking_tokens']:>4} tok   "
          f"forced={g['forced']}   answer={got!r}   {'CORRECT' if ok else 'wrong'}")
    print(render_trace(g))


## 6. The sweep machinery

One pass per (model, budget) over the item set. The per-item record is also the trace-bank schema
for pages 7 through 9, with `budget` where page 5 has `rung`:

```
problem_id, level, subject, question, ground_truth,
budget,                  # the cap this run used
trace, tokens,           # the text at this budget, and its token strings
cut_index, answer_start, # where the model stopped and where it resumed
truncated, forced,       # did it hit the cap; did we make it commit
think_closed, answer_closed,
extracted_answer, correct,
thinking_tokens, answer_tokens, tokens_generated,
model_id, run_id, carried_from
```

**Every budget is generated independently rather than sliced out of one long run.** A trace cut at
128 tokens and forced to answer is not the first 128 tokens of the 2,048-token trace followed by
its own conclusion, because the forced suffix changes what comes after it. That costs roughly 3x
the forward passes of slicing, and section 10 spends part of it checking the thing slicing would
have had to assume.

**One exception, and it is exact.** If the model stopped on its own at budget *b*, then at any
larger budget it walks the identical greedy path and stops in the identical place, because nothing
about the larger cap is visible to it. Those records carry forward with `carried_from` recorded,
and section 8 re-runs one from scratch to confirm it. This is why budgets are swept in ascending
order, and why anything counting *generations* rather than *records* must filter on
`carried_from is null`.

**Resume is free.** Records are appended as they are produced and keyed by
`(model, budget, problem)`, so an interrupted sweep picks up where it stopped.


In [ ]:
# The run id keys the resume cache, so it has to name everything that would change
# the numbers. Without dtype and device, a CPU/fp32 run and a later GPU/bf16 run over
# the same items would merge into one curve stitched from two numeric regimes.
RUN_ID = (f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}"
          f"-{str(DTYPE).rsplit('.', 1)[-1]}{'-4bit' if LOAD_IN_4BIT else ''}-{device.type}")


def load_done():
    '''(model_id, budget, problem_id) -> record, for resume.'''
    done = {}
    if TRACES_PATH.exists():
        with TRACES_PATH.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("run_id") == RUN_ID:
                    done[(r["model_id"], r["budget"], r["problem_id"])] = r
    return done


DONE = load_done()
print(f"run_id={RUN_ID}   {len(DONE)} generations already on disk")


def append_trace(rec):
    with TRACES_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def run_budget(model_id, mdl, tok, budget, rows, label=None):
    '''One pass over the items at one cap. Ascending budget order is required for
    the carry-forward below to be sound.'''
    records, n_generated, n_carried = [], 0, 0

    bar = tqdm(rows, desc=f"{label or f'budget {budget}':<22}", unit="item")
    for row in bar:
        key = (model_id, budget, row["unique_id"])
        if key in DONE:
            records.append(DONE[key])              # already on disk from an earlier run
            continue

        # Carry forward: an untruncated run at a smaller cap is bit-identical here.
        carried = None
        for smaller in [b for b in BUDGETS if b < budget][::-1]:
            prev = DONE.get((model_id, smaller, row["unique_id"]))
            if prev and not prev["truncated"]:
                carried = prev
                break

        if carried is not None:
            rec = dict(carried, budget=budget,
                       carried_from=carried.get("carried_from") or carried["budget"])
        else:
            g = run_at_budget(row["problem"], budget, mdl, tok)
            n_generated += 1
            extracted = extract_final_candidate(g["trace"])
            rec = {
                "run_id": RUN_ID,
                "model_id": model_id,
                "problem_id": row["unique_id"],
                "level": int(row["level"]),
                "subject": row["subject"],
                "question": row["problem"],
                "ground_truth": row["answer"],
                "budget": budget,
                "trace": g["trace"],
                "tokens": g["tokens"],
                "cut_index": g["cut_index"],
                "answer_start": g["answer_start"],
                "truncated": g["truncated"],
                "forced": g["forced"],
                "forced_suffix": g["forced_suffix"],
                "think_closed": g["think_closed"],
                "answer_closed": g["answer_closed"],
                "extracted_answer": extracted,
                "correct": bool(grade_answer(extracted, row["answer"])),
                "thinking_tokens": g["thinking_tokens"],
                "answer_tokens": g["answer_tokens"],
                "tokens_generated": g["tokens_generated"],
                "carried_from": None,
            }

        append_trace(rec)
        DONE[key] = rec
        records.append(rec)
        if rec.get("carried_from") is not None:
            n_carried += 1
        # gen counts real generations; resumed and carried-forward records are free,
        # so a bar that lumped them together would misreport what the sweep is doing.
        bar.set_postfix(acc=f"{mean([r['correct'] for r in records]):.2f}",
                        gen=n_generated, carried=n_carried)

    return records


## 7. Pilot: is there headroom?

Before the full sweep, one short run answers the question that can kill the page.

The figures need the model to **fail at low budget and succeed at high budget**. If it solves
everything at 2,048 tokens, figure 2 is flat at the top and figure 3's tail is empty; if it solves
nothing, both are flat at the bottom. So: run the largest budget over a handful of items and check
that accuracy lands in roughly the **40–70%** band.

**What to change if it does not** — decided now, from data, rather than argued about later:

- **above ~70%** — the item set is too easy for this model. Drop to level 5 only, or step the pair
  down (4B/1.7B → 1.7B/0.6B).
- **below ~40%** — the item set is too hard. Add level 3 back, or step the pair up.

The pilot also reports **mean thinking length at the largest budget**, which tells you whether the
budget grid spans the right range. If the model routinely stops well short of `MAX_BUDGET`, the top
of the grid is measuring nothing and the interesting action is further left.

These generations are the sweep's own, cached by `run_id`, so the pilot is free.


In [ ]:
PILOT_N = min(30 if RUN_MODE == "FULL" else 8, len(items))
pilot = run_budget(CFG["model_id"], model, tokenizer, MAX_BUDGET, items[:PILOT_N],
                   label=f"pilot @ {MAX_BUDGET}")

pilot_acc = mean([r["correct"] for r in pilot])
pilot_think = mean([r["thinking_tokens"] for r in pilot])
pilot_trunc = mean([r["truncated"] for r in pilot])

if pilot_acc > 0.70:
    verdict, advice = "TOO EASY", "drop to level 5 only, or step the model pair down a size"
elif pilot_acc < 0.40:
    verdict, advice = "TOO HARD", "add level 3 back, or step the model pair up a size"
else:
    verdict, advice = "in band", "proceed"

print(f"\naccuracy at budget {MAX_BUDGET}   {pilot_acc:.1%}  (n={PILOT_N}, want 40-70%)  -> {verdict}")
print(f"mean thinking length         {pilot_think:,.0f} tokens")
print(f"share still truncated at the top of the grid  {pilot_trunc:.0%}")
print(f"\n{advice}")
if PILOT_N < 20:
    print(f"\n(n={PILOT_N} resolves accuracy to {100/PILOT_N:.0f} percentage points at best, so read "
          f"this as a smoke\ntest rather than an estimate. FULL's pilot uses 30 items.)")

PILOT = dict(n=PILOT_N, budget=MAX_BUDGET, accuracy=pilot_acc,
             mean_thinking_tokens=pilot_think, truncation_rate=pilot_trunc,
             verdict=verdict, advice=advice)


## 8. The full sweep

Ascending budgets, one model at a time, the second model loaded only after the first is dropped.

Nothing is decided in this section — it produces the log that sections 10 to 13 read. The one thing
worth watching as it runs is the accuracy readout climbing across passes, which is figure 2 being
drawn one column at a time.


In [ ]:
# What you signed up for, projected from the warm-up. Generous: it assumes every
# generation runs to its cap, and most stop well short.
_n_models = 1 + (1 if CFG["second_model_id"] else 0)     # MODEL_IDS is built later
_worst = _n_models * len(items) * sum(BUDGETS)
print(f"up to {_worst:,} thinking tokens across {_n_models} model(s); at "
      f"~{WARMUP_TOK_S:.0f} tok/s that is under {_worst / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h "
      f"worst case, less whatever resumes from disk.\n")

results = {}          # model_id -> budget -> [records]
model_meta = {}       # model_id -> {label, params}


def sweep_model(model_id, mdl, tok, n_params, label):
    results[model_id] = {}
    model_meta[model_id] = dict(label=label, params=n_params, id=model_id,
                                dtype=str(next(mdl.parameters()).dtype),
                                tokenizer_sha256=tokenizer_fingerprint(tok))
    for b in BUDGETS:                                     # ascending: carry-forward needs it
        results[model_id][b] = run_budget(model_id, mdl, tok, b, items)
    return results[model_id]


PRIMARY = CFG["model_id"]
sweep_model(PRIMARY, model, tokenizer, N_PARAMS, label=PRIMARY.split("/")[-1])

# Carry-forward claims that an untruncated run at a smaller cap is bit-identical at a
# larger one. Check it here rather than in section 10, because by then this model has
# been freed to make room for the second one -- and section 10 is where FULL's data is
# validated, so a check that only ever runs in QUICK is not a check.
CARRY_CHECK = None
_c = next((r for b in BUDGETS for r in results[PRIMARY][b] if r.get("carried_from")), None)
if _c is not None:
    _again = run_at_budget(_c["question"], _c["budget"], model, tokenizer)
    CARRY_CHECK = dict(problem_id=_c["problem_id"], budget=_c["budget"],
                       carried_from=_c["carried_from"],
                       identical=_again["trace"] == _c["trace"])
    print(f"\ncarry-forward check: {_c['problem_id']} @ {_c['budget']} "
          f"(carried from {_c['carried_from']}) -> "
          f"{'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}")
    assert CARRY_CHECK["identical"], (
        "a carried-forward record does not reproduce when generated directly. Set "
        "CARRY_FORWARD aside, delete traces.jsonl and re-sweep before trusting anything below.")
else:
    print("\ncarry-forward check: nothing was carried forward in this run")


## 9. Asked nicely versus cut off

*The inset in figure 1's bottom right.*

Same items, same model, one nominal budget, two ways of imposing it:

- **asked nicely** — the prompt requests a budget in words (*use at most N tokens of reasoning*)
  and generation runs to the top of the grid;
- **cut off** — the hard cap at the same N, which is what the rest of this notebook does.

If the soft request overshoots, that is *why* the dial on this page is the cap rather than a
sentence in the prompt.

Read the token columns, not the accuracies: the soft arm runs to the top of the grid and the hard
arm is capped, so their accuracies are not comparable.


In [ ]:
SOFT_N = min(20, len(items))
SOFT_BUDGET = BUDGETS[len(BUDGETS) // 2]         # a middle rung of the grid

# A plain brevity instruction, not "use at most N tokens": a model cannot count its
# own tokens as it emits them, so a numeric self-budget would be an unfair request
# and would make the overshoot finding too easy. This is the kind of thing a person
# would actually write.
SOFT_REQUEST = "\n\nKeep your reasoning to one or two sentences, then give the final answer."


def median(xs):
    xs = sorted(xs)
    m = len(xs) // 2
    return xs[m] if len(xs) % 2 else (xs[m - 1] + xs[m]) / 2


soft = []
for k, row in enumerate(items[:SOFT_N], start=1):
    key = ("soft", SOFT_BUDGET, row["unique_id"])
    if key in DONE:
        rec = DONE[key]
    else:
        g = run_at_budget(row["problem"] + SOFT_REQUEST, MAX_BUDGET, model, tokenizer)
        got = extract_final_candidate(g["trace"])
        rec = {"run_id": RUN_ID, "model_id": "soft", "problem_id": row["unique_id"],
               "level": int(row["level"]), "subject": row["subject"],
               "question": row["problem"], "ground_truth": row["answer"],
               "budget": SOFT_BUDGET, "arm": "asked_nicely", "request": SOFT_REQUEST.strip(),
               "trace": g["trace"], "tokens": g["tokens"],
               "cut_index": g["cut_index"], "answer_start": g["answer_start"],
               "truncated": g["truncated"], "forced": g["forced"],
               "forced_suffix": g["forced_suffix"], "think_closed": g["think_closed"],
               "answer_closed": g["answer_closed"], "extracted_answer": got,
               "correct": bool(grade_answer(got, row["answer"])),
               "thinking_tokens": g["thinking_tokens"], "answer_tokens": g["answer_tokens"],
               "tokens_generated": g["tokens_generated"], "carried_from": None}
        append_trace(rec)            # the inset ships in the download tier too
        DONE[key] = rec
    soft.append(rec)
    print(f"\rasked nicely: {k}/{SOFT_N}", end="")
print()

soft_ids = {r["problem_id"] for r in soft}
hard_runs = [r for r in results[PRIMARY][SOFT_BUDGET] if r["problem_id"] in soft_ids]

INSET = {
    "nominal_budget": SOFT_BUDGET,
    "n": SOFT_N,
    "request": SOFT_REQUEST.strip(),
    "soft": dict(mean_thinking_tokens=mean([r["thinking_tokens"] for r in soft]),
                 median_thinking_tokens=median([r["thinking_tokens"] for r in soft]),
                 over_budget_rate=mean([r["thinking_tokens"] > SOFT_BUDGET for r in soft]),
                 measured_at_budget=MAX_BUDGET,
                 accuracy=mean([r["correct"] for r in soft])),
    "hard": dict(mean_thinking_tokens=mean([r["thinking_tokens"] for r in hard_runs]),
                 median_thinking_tokens=median([r["thinking_tokens"] for r in hard_runs]),
                 over_budget_rate=0.0,
                 measured_at_budget=SOFT_BUDGET,
                 accuracy=mean([r["correct"] for r in hard_runs])),
}
# Against the NOMINAL budget, which is the number both arms were asked for. Dividing
# by the hard arm's realised mean would flatter the comparison, since that arm cannot
# exceed the cap by construction.
INSET["overshoot_multiple"] = INSET["soft"]["mean_thinking_tokens"] / SOFT_BUDGET

print(f"\nnominal budget: {SOFT_BUDGET} tokens, n={SOFT_N}\n")
print(f"{'':<14}{'mean tok':>10}{'median':>9}{'over budget':>13}{'accuracy':>10}")
for k in ("soft", "hard"):
    d = INSET[k]
    print(f"{'asked nicely' if k == 'soft' else 'cut off':<14}"
          f"{d['mean_thinking_tokens']:>10,.0f}{d['median_thinking_tokens']:>9,.0f}"
          f"{d['over_budget_rate']:>12.0%}{d['accuracy']:>10.0%}")
print(f"\nasked nicely runs {INSET['overshoot_multiple']:.1f}x the nominal {SOFT_BUDGET}-token "
      f"budget; the cap runs 1.0x by construction")
print("the two accuracy figures are NOT comparable: the soft arm was free to run to "
      f"{MAX_BUDGET}\ntokens, the hard arm was capped at {SOFT_BUDGET}. The columns to read are the "
      "token columns.")


In [ ]:
# The second curve. Loaded only now, so the two models are never resident together.
SECOND = CFG["second_model_id"]
if SECOND:
    del model                                 # the primary is done; free it before loading the next
    PRIMARY_LOADED = False
    free_memory()

    model2, tokenizer2, n2 = load_model(SECOND)
    sweep_model(SECOND, model2, tokenizer2, n2, label=SECOND.split("/")[-1])
    del model2
    free_memory()
else:
    print("QUICK mode: one curve only. Figure 2 wants two model sizes -- the invariant is "
          "what survives the difference in level -- so run FULL before drawing the page's figure.")

MODEL_IDS = [m for m in (PRIMARY, SECOND) if m in results]
print("swept:", ", ".join(MODEL_IDS))


## 10. Sweep integrity

Six checks the page's honesty rests on — four in the next cell, two in the one after. If any of them
fails, the figures below do not mean what the page says they mean.

Checks **3** and **5** are the ones that matter most, and they are specific to this page:

- **The cut is real, and it is where we say it is.** Every trace at budget *b* must contain at most
  *b* tokens of the model's own thinking. This is the assertion that the x-axis is what it claims.
- **The trace at a small budget is a genuine prefix of the trace at a large one.** Under greedy
  decoding it must be — the model cannot see the cap. We check it rather than assume it, because
  figure 1 leans on it: the page greys out what the model *would* have written past the cut, and
  that greyed text is only honest if the visible text really is the beginning of it. This is also
  the check that licenses the carry-forward optimisation in section 6.


In [ ]:
all_records = [r for mid in MODEL_IDS for b in BUDGETS for r in results[mid][b]]
forced_records = [r for r in all_records if r["forced"]]

print(f"{len(all_records):,} records  ({len(forced_records):,} forced)\n")

# 1. every trace reconstructs from its token strings
bad = [r for r in all_records if "".join(r["tokens"]) != r["trace"]]
print(f"1. traces reconstruct from tokens ......... {len(bad)} violations")
assert not bad, "a trace's token strings do not join back to its text; figure 1's reveal would lie"

# 2. the forced answer had room to close its brace
unclosed = [r for r in forced_records if r["answer_closed"] is False]
rate = len(unclosed) / max(len(forced_records), 1)
print(f"2. forced answers closed the brace ........ {1 - rate:.1%} "
      f"({len(unclosed)} of {len(forced_records)} ran out of allowance)")
if rate > 0.05:
    print(f"   ^ above 5%: raise ANSWER_ALLOWANCE (now {ANSWER_ALLOWANCE}) and re-run. An unclosed "
          f"brace is not\n     merely graded wrong -- get_last_boxed returns nothing, so "
          f"extract_final_candidate falls\n     back to the last NUMBER anywhere in the trace, "
          f"which can score right or wrong by luck.\n     Those items are graded by a different "
          f"rule than the rest, which is ours, not the model's.")

# 3. the cap actually bound where it says it did
over = [r for r in all_records if r["thinking_tokens"] > r["budget"]]
print(f"3. thinking never exceeded its budget ..... {len(over)} violations")
assert not over, "a trace thought for longer than its cap; the x-axis does not mean what it says"

# 4. the dial bites at the bottom of the grid and lets go at the top
print("\n4. how often the cap bound, by budget:")
print(f"   {'budget':>8}" + "".join(f"{mid.split('/')[-1]:>16}" for mid in MODEL_IDS))
for b in BUDGETS:
    row = "".join(f"{mean([r['truncated'] for r in results[mid][b]]):>15.0%} " for mid in MODEL_IDS)
    print(f"   {b:>8}" + row)
print("   (falling to zero at the top of the grid means the largest budget is no longer a")
print("    constraint -- past that point the page's dial is not doing anything)")


In [ ]:
# 5. a small-budget trace is a genuine prefix of a large-budget one
def prefix_len(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


prefix_report = []
for mid in MODEL_IDS:
    ref = {r["problem_id"]: r for r in results[mid][MAX_BUDGET]}
    for b in BUDGETS[:-1]:
        for r in results[mid][b]:
            R = ref[r["problem_id"]]
            own = r["tokens"][: r["cut_index"]] if r["forced"] else r["tokens"]
            ref_own = R["tokens"][: R["cut_index"]] if R["forced"] else R["tokens"]
            n = prefix_len(own, ref_own)
            prefix_report.append(dict(model_id=mid, budget=b, problem_id=r["problem_id"],
                                      shared=n, own=len(own), exact=n == len(own),
                                      # only a truncated record is an INDEPENDENT generation;
                                      # an untruncated one is the same record the carry-forward
                                      # copied to MAX_BUDGET, so comparing it proves nothing
                                      independent=r["truncated"]))

indep = [p for p in prefix_report if p["independent"]]
exact = mean([p["exact"] for p in indep])
mismatches = sum(not p["exact"] for p in indep)
print(f"5. short traces are prefixes of long ones . {exact:.1%} exact "
      f"({mismatches} of {len(indep)} diverge)")
print(f"   {len(prefix_report) - len(indep)} of {len(prefix_report)} comparisons excluded as "
      f"self-comparisons (the\n   record at the larger budget IS the record at the smaller one, "
      f"carried forward)")
if not indep:
    print("   nothing independent to compare: no record was truncated below the top budget")
elif exact < 1.0:
    worst = min(prefix_report, key=lambda p: p["shared"] / max(p["own"], 1))
    print(f"   worst case shares {worst['shared']}/{worst['own']} tokens "
          f"({worst['model_id']} @ {worst['budget']}, {worst['problem_id']})")
    print("   Some divergence is expected: greedy argmax is only deterministic up to floating-point")
    print("   reduction order, which shifts with sequence length on some kernels. A few near-total")
    print("   prefixes are fine. Widespread short ones mean do_sample slipped back on -- check that")
    print("   first, because everything downstream assumes the dial is the only variable.")

# 6. carry-forward is exact, not an approximation -- re-run one and compare
carried = [r for r in all_records if r.get("carried_from")]
print(f"\n6. records carried forward from a smaller budget: {len(carried)} of {len(all_records)}")
if CARRY_CHECK is None:
    print("   no spot check: nothing was carried forward on the primary model")
else:
    print(f"   spot check (run in section 8, while the primary was still loaded): "
          f"{CARRY_CHECK['problem_id']} @ {CARRY_CHECK['budget']} "
          f"-> {'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}")

INTEGRITY = dict(n_records=len(all_records), n_forced=len(forced_records),
                 unclosed_forced_rate=rate, prefix_exact_rate=exact,
                 n_prefix_comparisons=len(indep), n_carried_forward=len(carried),
                 carry_forward_spot_check=CARRY_CHECK)


## 11. Beat 1 — the truncation view

*Figure 1 on the page.* One real problem, every budget, the trace visibly cut where the reader put
the dial.

The specimen is **curated from the sweep, not generated for the figure**: a problem that is wrong
at low budget and right at high budget, with the flip near the middle of the grid so the reader has
room to drag both ways, and among those the shortest traces, because legibility is the binding
constraint rather than existence.

Three fields in the record below are what the figure draws with. `reference_tokens` is the full
trace the page greys out past the cut, and check 5 in section 10 is what makes showing it honest.
`answer_start` is where the tint changes. And the per-budget `correct` flags are what make the
verdict chip flip discontinuously, which is the contrast with figure 2's smooth curve.


In [ ]:
def by_problem(mid):
    '''problem_id -> {budget: record}'''
    out = {}
    for b in BUDGETS:
        for r in results[mid][b]:
            out.setdefault(r["problem_id"], {})[b] = r
    return out


PRIMARY_BY_PROBLEM = by_problem(PRIMARY)


def first_correct(row):
    for b in BUDGETS:
        if row[b]["correct"]:
            return b
    return None


candidates = []
for pid, row in PRIMARY_BY_PROBLEM.items():
    flip = first_correct(row)
    if flip is None or flip == BUDGETS[0]:
        continue                                   # need a visible "before"
    if not row[BUDGETS[BUDGETS.index(flip) - 1]]["truncated"]:
        continue                                   # the budget below must show a real cut
    centre = abs(BUDGETS.index(flip) - (len(BUDGETS) - 1) / 2)
    candidates.append((centre, len(row[MAX_BUDGET]["tokens"]), pid, flip))

candidates.sort()
print(f"{len(candidates)} problems flip above the first budget with a visible cut below it")

SPECIMEN_IS_FALLBACK = False
if candidates:
    _, _, SPECIMEN_ID, SPECIMEN_FLIP = candidates[0]
else:
    # A small grid on a small model can miss this by chance. Degrading with a flag beats
    # failing the run: figure 2 and figure 3 do not depend on the specimen, and QUICK
    # exists to deliver those. A flagged specimen must not be translated to the page.
    SPECIMEN_IS_FALLBACK = True
    ranked = sorted(
        PRIMARY_BY_PROBLEM.items(),
        key=lambda kv: (first_correct(kv[1]) is None,          # prefer one that flips at all
                        not kv[1][BUDGETS[0]]["truncated"],    # then one with a visible cut
                        len(kv[1][MAX_BUDGET]["tokens"]),
                        kv[0]))
    SPECIMEN_ID = ranked[0][0]
    SPECIMEN_FLIP = first_correct(ranked[0][1])
    print("\nWARNING: no problem is wrong at a low budget and right at a high one with a cut\n"
          "         below the flip. Every problem is solved at the smallest budget, or at\n"
          "         none. That is the pilot's too-easy / too-hard verdict reappearing.\n"
          "         Falling back to an illustrative specimen; derived.json flags it, and it\n"
          "         must not be translated to the page. Figures 2 and 3 are unaffected.\n")
specimen = PRIMARY_BY_PROBLEM[SPECIMEN_ID]
print(f"specimen: {SPECIMEN_ID}  (level {specimen[MAX_BUDGET]['level']}, flips at {SPECIMEN_FLIP})")


In [ ]:
print(textwrap.fill("Q: " + specimen[MAX_BUDGET]["question"], 96))
print("ground truth:", specimen[MAX_BUDGET]["ground_truth"])

for b in BUDGETS:
    r = specimen[b]
    print("\n" + "=" * 100)
    print(f"budget {b:>5}   thinking {r['thinking_tokens']:>4} tok   forced={r['forced']!s:<5}  "
          f"answer={r['extracted_answer']!r}   {'CORRECT' if r['correct'] else 'wrong'}"
          f"{'   <- flips here' if b == SPECIMEN_FLIP else ''}")
    print(render_trace(r))

verdicts = "".join("C" if specimen[b]["correct"] else "." for b in BUDGETS)
print(f"\nverdict across the grid: {verdicts}   (a step, not a ramp -- that is beat 1's whole point)")


## 12. Beat 2 — the population curve

*Figure 2 on the page.* Share of problems solved against token budget, log x-axis, two model sizes.

**If the curve is not monotone, keep it.** A dip at some middle budget, where the model has room to
talk itself out of a right answer but not enough to recover, is a real result and page 17 is
waiting for it. The cell after the plot checks for exactly this so it cannot be smoothed away by
accident.


In [ ]:
def curve_for(mid):
    pts = []
    for b in BUDGETS:
        rs = results[mid][b]
        pts.append(dict(
            budget=b,
            n=len(rs),
            n_correct=sum(r["correct"] for r in rs),
            accuracy=mean([r["correct"] for r in rs]),
            mean_tokens=mean([r["tokens_generated"] for r in rs]),
            total_tokens=sum(r["tokens_generated"] for r in rs),
            truncation_rate=mean([r["truncated"] for r in rs]),
            forced_rate=mean([r["forced"] for r in rs]),
        ))
    return pts


CURVES = {mid: curve_for(mid) for mid in MODEL_IDS}

for mid in MODEL_IDS:
    print(f"\n{mid}   n={CURVES[mid][0]['n']} items, MATH-500 levels 4-5, greedy")
    print(f"  {'budget':>8}{'solved':>10}{'accuracy':>10}{'mean tok':>10}{'total tok':>12}{'cap bound':>11}")
    for p in CURVES[mid]:
        print(f"  {p['budget']:>8}{p['n_correct']:>6}/{p['n']:<3}{p['accuracy']:>9.1%}"
              f"{p['mean_tokens']:>10,.0f}{p['total_tokens']:>12,}{p['truncation_rate']:>11.0%}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
COLOURS = ["#3b4a6b", "#a2704a"]

for (mid, pts), colour in zip(CURVES.items(), COLOURS):
    ax.plot([p["budget"] for p in pts], [p["accuracy"] * 100 for p in pts],
            marker="o", ms=4.5, lw=2, color=colour, label=model_meta[mid]["label"])

ax.set_xscale("log", base=2)
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget (max_new_tokens inside the think block)")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
# On the page the y-axis is deemphasised in favour of the curve's shape; here it stays
# labelled, because the notebook is where you check the level and the page is where you
# read the shape.
ax.set_title(f"MATH-500 levels 4-5, n={CURVES[PRIMARY][0]['n']}, greedy, budget forced at the cap",
             fontsize=9, loc="left", color="#555")

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_curve.png", dpi=160)
plt.show()


In [ ]:
# Write down what the run produced before deciding what the caption says about it.
KNEE_GAIN_PTS = 2.0
BEND = {}
for mid in MODEL_IDS:
    pts = CURVES[mid]
    gains = [dict(from_budget=pts[i - 1]["budget"], to_budget=pts[i]["budget"],
                  gain_pts=(pts[i]["accuracy"] - pts[i - 1]["accuracy"]) * 100)
             for i in range(1, len(pts))]
    monotone = all(g["gain_pts"] >= 0 for g in gains)
    # "The bend" = the first doubling after which no later doubling buys KNEE_GAIN_PTS.
    # 2 points is a judgement call, chosen as "smaller than the page could honestly
    # draw a distinction at". It is also resolution-dependent: at FULL's n=160 one
    # problem is 0.6 points, so 2 points is a real threshold; at QUICK's n=20 one
    # problem is 5 points, so it degenerates to "accuracy never rises again". Read
    # the gains table below rather than the single number if the two disagree.
    knee = None
    for i, g in enumerate(gains):
        if all(h["gain_pts"] < KNEE_GAIN_PTS for h in gains[i:]):
            knee = g["from_budget"]
            break
    BEND[mid] = dict(gains=gains, monotone=monotone, knee_budget=knee,
                     ceiling=max(p["accuracy"] for p in pts),
                     floor=pts[0]["accuracy"],
                     total_range_pts=(max(p["accuracy"] for p in pts) - pts[0]["accuracy"]) * 100)

    print(f"\n{mid}")
    for g in gains:
        bar = "#" * max(int(round(g["gain_pts"])), 0)
        print(f"  {g['from_budget']:>5} -> {g['to_budget']:<5} {g['gain_pts']:>+6.1f} pts  {bar}")
    print(f"  floor {pts[0]['accuracy']:.0%} at {pts[0]['budget']} tokens, "
          f"ceiling {BEND[mid]['ceiling']:.0%}, span {BEND[mid]['total_range_pts']:.0f} pts")
    print(f"  bends at: {knee if knee else 'has not bent inside this grid -- extend it'}")
    if not monotone:
        dips = [g for g in gains if g["gain_pts"] < 0]
        print(f"  NON-MONOTONE: {len(dips)} doubling(s) lose accuracy "
              f"({', '.join(str(d['to_budget']) for d in dips)}).")
        print("  Keep it and show it. A model with room to talk itself out of a right answer but")
        print("  not enough room to recover is a real finding, and page 17 is already waiting for it.")


## 13. Beat 3 — the flip-point strip

*Figure 3 on the page.* One row per problem, sorted by flip point, shaded from the budget where it
first grades correct.

Two things the strip shows that the curve cannot. A problem has a **threshold**, not a gradient, so
each row is a step and figure 2's smooth climb is the average of a few hundred steps landing in
different places. And thresholds are **not concentrated**: the histogram below has a long right
tail that runs off the end of the grid.

The strip also makes per-item **regressions** visible, a problem correct at 128 and wrong at 256.
They are marked rather than hidden, and they are the honest reason flip point is defined as *first*
correct rather than *permanently* correct.


In [ ]:
FLIP_MODEL = PRIMARY
rows = by_problem(FLIP_MODEL)

flip_items = []
for pid, row in rows.items():
    verdicts = [bool(row[b]["correct"]) for b in BUDGETS]
    fc = next((BUDGETS[j] for j, v in enumerate(verdicts) if v), None)
    # "monotone" = once it flips, it stays flipped. Note the slice has to start at the
    # flip, not at a loop variable: a generator expression's index does not leak out.
    stable = all(verdicts[BUDGETS.index(fc):]) if fc is not None else True
    flip_items.append(dict(
        problem_id=pid,
        level=row[MAX_BUDGET]["level"],
        subject=row[MAX_BUDGET]["subject"],
        correct_by_budget=verdicts,
        first_correct_budget=fc,
        monotone=stable,
        tokens_at_max=row[MAX_BUDGET]["tokens_generated"],
        truncated_at_max=row[MAX_BUDGET]["truncated"],
    ))

# never-flips sort to the bottom, then by flip point, then by how long they took
flip_items.sort(key=lambda d: (d["first_correct_budget"] is None,
                               BUDGETS.index(d["first_correct_budget"]) if d["first_correct_budget"] else 0,
                               d["tokens_at_max"]))

never = [d for d in flip_items if d["first_correct_budget"] is None]
regress = [d for d in flip_items if not d["monotone"]]
hist = {b: sum(d["first_correct_budget"] == b for d in flip_items) for b in BUDGETS}

print(f"{FLIP_MODEL}, n={len(flip_items)}\n")
print(f"  {'flips at':>10}{'problems':>10}   distribution")
for b in BUDGETS:
    print(f"  {b:>10}{hist[b]:>10}   {'#' * hist[b]}")
print(f"  {'never':>10}{len(never):>10}   {'#' * len(never)}")
print(f"\n{len(never)} of {len(flip_items)} ({len(never)/len(flip_items):.0%}) are never reached by any "
      f"budget in this grid.")
print(f"{len(regress)} problem(s) go correct and then wrong again at a larger budget.")


In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

grid = np.array([[1 if v else 0 for v in d["correct_by_budget"]] for d in flip_items])

fig, (ax, axh) = plt.subplots(1, 2, figsize=(10, 5), sharey=False,
                              gridspec_kw=dict(width_ratios=[3, 1], wspace=0.25))

ax.imshow(grid, aspect="auto", interpolation="nearest",
          cmap=ListedColormap(["#e8e6e1", "#3b4a6b"]), vmin=0, vmax=1)
ax.set_xticks(range(len(BUDGETS)))
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget")
ax.set_ylabel(f"problems, sorted by flip point  (n={len(flip_items)})")
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# regressions: correct, then wrong again at a larger budget
for y, d in enumerate(flip_items):
    if not d["monotone"]:
        for x, v in enumerate(d["correct_by_budget"]):
            if not v and any(d["correct_by_budget"][:x]):
                ax.plot(x, y, marker="x", ms=4, color="#b23", mew=1.2)

if never:
    ax.axhline(len(flip_items) - len(never) - 0.5, color="#b23", lw=1.2, ls="--")
    ax.text(len(BUDGETS) - 0.4, len(flip_items) - len(never) / 2,
            f"never flips\n{len(never)} problems", color="#b23", fontsize=8, va="center")

axh.barh(range(len(BUDGETS)), [hist[b] for b in BUDGETS], color="#3b4a6b", height=0.6)
axh.barh([len(BUDGETS)], [len(never)], color="#b23", height=0.6)
axh.set_yticks(range(len(BUDGETS) + 1))
axh.set_yticklabels([str(b) for b in BUDGETS] + ["never"], fontsize=8)
axh.invert_yaxis()
axh.set_xlabel("problems")
axh.set_title("flip point", fontsize=9, loc="left", color="#555")
axh.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"{model_meta[FLIP_MODEL]['label']} — the budget at which each problem first grades correct",
             fontsize=10, x=0.02, ha="left", color="#333")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_flip_strip.png", dpi=160)
plt.show()


## 14. The handoff to page 7

Page 6 spends its whole budget on **one** chain, and figure 3 shows a band that chain never
reaches. So: same total budget, spent differently. Instead of one chain of `MAX_BUDGET` tokens, N
chains of `BUDGETS[0]`.

The numbers below are the bridge, and page 7's run is measured against *this* budget, which is what
makes it a comparison rather than a new experiment.


In [ ]:
top = next(p for p in CURVES[PRIMARY] if p["budget"] == MAX_BUDGET)
bot = CURVES[PRIMARY][0]
n_chains = MAX_BUDGET // BUDGETS[0]

HANDOFF = {
    "model_id": PRIMARY,
    "max_budget": MAX_BUDGET,
    "min_budget": BUDGETS[0],
    "chains_at_min_budget": n_chains,
    "accuracy_at_max": top["accuracy"],
    "accuracy_at_min": bot["accuracy"],
    "total_tokens_at_max": top["total_tokens"],
    "mean_tokens_at_max": top["mean_tokens"],
    "never_flip_share": len(never) / len(flip_items),
    "never_flip_count": len(never),
    "n_items": len(flip_items),
}

print(f"one chain of {MAX_BUDGET:,} tokens     -> {top['accuracy']:.0%} solved "
      f"({top['mean_tokens']:,.0f} tokens/problem actually spent)")
print(f"one chain of {BUDGETS[0]:,} tokens      -> {bot['accuracy']:.0%} solved")
print(f"\nthe same budget buys {n_chains} chains of {BUDGETS[0]} tokens. Page 7 asks what {n_chains} "
      f"voters at {bot['accuracy']:.0%} each are worth.")
print(f"\nand {len(never)} of {len(flip_items)} problems ({len(never)/len(flip_items):.0%}) are not "
      f"reached by any budget here, so length is not the only axis.")


## 15. Artifacts

`derived.json` is the contract between this notebook and the page: the page reads it and nothing
else, and **no inference runs at demonstration time.**

**Which traces get bundled.** Figure 1's specimen at every budget, plus `N_BUNDLED_ROWS` more
problems spanning the flip-point range (early flippers, late flippers, never-flippers), because
figure 3's rows are clickable and a click loads that problem into figure 1. Everything else stays
in the download tier. The size report at the end keeps this a decision rather than a drift: if
`derived.json` creeps past a megabyte, lower `N_BUNDLED_ROWS`.

The **run manifest** is what makes "re-derivable" checkable: model revision hashes, tokenizer hash,
dtype, device, library versions, seed, item-set hash, notebook git sha, timestamp, run mode.


In [ ]:
N_BUNDLED_ROWS = 8          # figure 3's clickable rows, spanning the flip-point range


def bundle_row(pid):
    row = PRIMARY_BY_PROBLEM[pid]
    ref = row[MAX_BUDGET]
    return {
        "problem_id": pid,
        "level": ref["level"],
        "subject": ref["subject"],
        "question": ref["question"],
        "ground_truth": ref["ground_truth"],
        "first_correct_budget": first_correct(row),
        # what the model would have written past the cut -- figure 1's greyed text
        "reference_tokens": ref["tokens"][: ref["cut_index"]] if ref["forced"] else ref["tokens"],
        "reference_truncated": ref["truncated"],
        "by_budget": [
            {k: row[b][k] for k in ("budget", "trace", "tokens", "cut_index", "answer_start",
                                    "truncated", "forced", "forced_suffix", "answer_closed",
                                    "extracted_answer", "correct", "thinking_tokens",
                                    "answer_tokens", "tokens_generated")}
            for b in BUDGETS
        ],
    }


# span the sorted strip so the bundled rows include early, late and never-flipping problems
idx = sorted({round(i * (len(flip_items) - 1) / max(N_BUNDLED_ROWS - 1, 1))
              for i in range(N_BUNDLED_ROWS)})
bundled_ids = [flip_items[i]["problem_id"] for i in idx]
if SPECIMEN_ID not in bundled_ids:
    bundled_ids.insert(0, SPECIMEN_ID)

print(f"bundling {len(bundled_ids)} problems x {len(BUDGETS)} budgets:")
for pid in bundled_ids:
    fc = first_correct(PRIMARY_BY_PROBLEM[pid])
    print(f"  {pid:<34} flips at {str(fc) if fc else 'never':>6}"
          f"{'   <- figure 1 specimen' if pid == SPECIMEN_ID else ''}")


In [ ]:
def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True, check=True).stdout.strip()
    except Exception:
        return None


def repo_revision(model_id):
    try:
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception as e:
        print(f"WARNING: could not resolve the repo revision for {model_id} ({e}). The "
              f"manifest will say null, and the run is then not pinned to specific weights "
              f"-- which is the property section 0 promises. Re-run this cell online.")
        return None


def notebook_sha256():
    '''Hash the notebook file itself. The git sha alone can name a commit that does
    not contain this notebook (uncommitted edits, or a dirty tree), so it is not on
    its own a pin on the code that produced the run.'''
    here = Path("page_06_thinking_longer.ipynb")
    return hashlib.sha256(here.read_bytes()).hexdigest() if here.exists() else None


manifest = {
    "run_id": RUN_ID,
    "notebook": "page_06_thinking_longer.ipynb",
    "page": 6,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": git_sha(),
    "notebook_sha256": notebook_sha256(),
    "models": [
        {"id": mid,
         "label": model_meta[mid]["label"],
         "params": model_meta[mid]["params"],
         "revision": repo_revision(mid),
         "dtype": model_meta[mid]["dtype"],              # as loaded, not as requested
         "tokenizer_sha256": model_meta[mid]["tokenizer_sha256"],
         "role": "primary" if mid == PRIMARY else "second curve",
         "thinking": "on"}
        for mid in MODEL_IDS
    ],
    "dtype_requested": str(DTYPE),
    "load_in_4bit": LOAD_IN_4BIT,
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "temperature": None, "top_p": None, "top_k": None},
    # greedy argmax is only deterministic up to floating-point reduction order, and
    # ch02's get_device() enables TF32 on CUDA, so this belongs in the record.
    "tf32_matmul": bool(getattr(getattr(torch.backends, "cuda", None), "matmul", None)
                        and torch.backends.cuda.matmul.allow_tf32),
    "budgets": BUDGETS,
    "forcing": {"close": FORCE_CLOSE, "stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE},
    "items": {
        "source": "MATH-500 (ch03 copy)",
        "filter": "level in {4, 5}",
        "n": len(items),
        "item_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "prompt_template": render_prompt("{question}"),
    "chat_template_thinking_on": on,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": INTEGRITY,
}

derived = {
    "manifest": manifest,
    "budgets": BUDGETS,
    # The one line printed under figure 2. It is what makes the figure re-derivable
    # rather than merely reproducible, and it is worth more than any sentence
    # describing a curve the reader can already see.
    "procedure": (
        f"Cap generation at N tokens; if the model is still inside its think block, "
        f"close it and append \"Final answer: \\boxed{{\"; allow {ANSWER_ALLOWANCE} more "
        f"tokens; grade with MATH-500's reference answer. Repeat for "
        f"N in {{{', '.join(str(b) for b in BUDGETS)}}} over {len(items)} problems "
        f"from levels 4-5, greedy throughout."),
    "pilot": PILOT,
    "figure_1": {
        "specimen_id": SPECIMEN_ID,
        "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
        "flip_budget": SPECIMEN_FLIP,
        "n_candidate_specimens": len(candidates),
        "rows": [bundle_row(pid) for pid in bundled_ids],
        "inset": INSET,
    },
    "figure_2": {
        "curves": [{"model_id": mid,
                    "label": model_meta[mid]["label"],
                    "params": model_meta[mid]["params"],
                    "points": CURVES[mid],
                    **BEND[mid]}
                   for mid in MODEL_IDS],
    },
    "figure_3": {
        "model_id": FLIP_MODEL,
        "items": flip_items,
        "histogram": {str(b): hist[b] for b in BUDGETS},
        "never_flip_count": len(never),
        "never_flip_share": len(never) / len(flip_items),
        "n_regressions": len(regress),
        # the rows whose full traces are bundled: clicking one loads it into figure 1
        "clickable_row_ids": bundled_ids,
    },
    "handoff": HANDOFF,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

kb = lambda p: p.stat().st_size / 1024
print(f"derived.json   {kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {kb(TRACES_PATH):>10,.1f} KB   <- published as a download "
      f"({len(carried)} of {len(all_records)} records are carry-forward duplicates)")

if kb(DERIVED_PATH) > 1024:
    print("\n^ derived.json is over 1 MB. Lower N_BUNDLED_ROWS: the page bundles what it draws "
          "and links the rest.")
if kb(TRACES_PATH) > 50 * 1024:
    print("\n^ traces.jsonl is over 50 MB, which is a download rather than an asset. Page 7's "
          "N-sweep\n  will be larger again; if this is uncomfortable now, that is the signal to "
          "compress or to\n  store carry-forward records as pointers before page 7 forces it.")

print(f"\nNot committed by this notebook: out/ is gitignored. Promoting a run to the page is "
      f"pipeline\nstep 4 -- copy derived.json and manifest.json into web/data/, where the "
      f"cross-language\nagreement test recomputes these same aggregates in TypeScript and asserts "
      f"they match.")


## 16. Where to check what this run may claim

The claims contract lives in [`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §7, beside the
rest of the page's design, and the translation step checks itself against it.

Two of its gates can only be answered after a `FULL` run, and both were printed above: whether the
curve bends inside the grid (`figure_2.knee_budget`), and whether the never-flipping band is big
enough to see and small enough to be a band (`figure_3.never_flip_share`).
